In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" tqdm
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes transformers datasets


In [ ]:
# Cell 2: Load Qwen3-8B 4-bit bằng Unsloth
import os
import json
import glob
import torch
from tqdm import tqdm
from unsloth import FastLanguageModel


# Nếu có nhiều file, lấy file đầu tiên. Có thể sửa thủ công nếu cần.
INPUT_FILE = "/kaggle/input/datasets/hdtuznn/crawl-data-medical/rag_chunks.jsonl"
OUTPUT_FILE = "/kaggle/working/rag_chunks_vi.jsonl"
CHECKPOINT_FILE = "/kaggle/working/rag_chunks_vi.partial.jsonl"

print("INPUT_FILE:", INPUT_FILE)
print("OUTPUT_FILE:", OUTPUT_FILE)

MODEL_NAME = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = """
You are a professional medical translator.

Translate ONLY the given medical content from English to Vietnamese.

Rules:
- Keep drug names unchanged.
- Keep disease names medically accurate.
- Keep dosage units unchanged.
- Keep numbers unchanged.
- Keep abbreviations unchanged when appropriate, such as INR, CYP2C9, ECG, FDA.
- Do not translate metadata.
- Do not add explanation.
- Do not add notes.
- Return only the Vietnamese translation.
"""

def clean_translation(text: str) -> str:
    text = text.strip()
    # Phòng trường hợp model lỡ thêm nhãn
    prefixes = [
        "Vietnamese translation:",
        "Translation:",
        "Bản dịch tiếng Việt:",
        "Dịch tiếng Việt:",
    ]
    for p in prefixes:
        if text.lower().startswith(p.lower()):
            text = text[len(p):].strip()
    return text

def translate_content(text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.strip()},
        {"role": "user", "content": text.strip()},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,   # Qwen3: tắt thinking để output chỉ là bản dịch
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1536,
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    translated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

    return clean_translation(translated)


In [ ]:
# Cell 3: Dịch thử 1 chunk trước để kiểm tra chất lượng
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    first_item = json.loads(next(f))

print("Original content:")
print(first_item.get("content", "")[:1000])

print("\nTranslated content:")
print(translate_content(first_item.get("content", ""))[:1000])


In [ ]:
# Cell 4: Translate toàn bộ file JSONL
# Chỉ thay item["content"], metadata giữ nguyên.

translated_count = 0
skipped_count = 0

with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(CHECKPOINT_FILE, "w", encoding="utf-8") as fout:

    for line in tqdm(fin, desc="Translating chunks"):
        item = json.loads(line)

        content = item.get("content", "")
        if isinstance(content, str) and content.strip():
            try:
                item["content"] = translate_content(content)
                translated_count += 1
            except Exception as e:
                print("Translate failed for id:", item.get("id", "unknown"))
                print("Error:", repr(e))
                # Giữ content gốc nếu lỗi để không mất dữ liệu
                skipped_count += 1
        else:
            skipped_count += 1

        fout.write(json.dumps(item, ensure_ascii=False) + "\n")
        fout.flush()

# Đổi tên partial thành file hoàn chỉnh
os.replace(CHECKPOINT_FILE, OUTPUT_FILE)

print("Done!")
print("Translated chunks:", translated_count)
print("Skipped/failed chunks:", skipped_count)
print("Saved to:", OUTPUT_FILE)
